# Plots and ablations

Loss curves, latent representations, decoded digits, confusion maps and the experiments from Sections 6.3–6.7. Longer comparisons are separate switches below. All figures come from the current run.


In [ ]:
%run ./functions.ipynb
from copy import deepcopy
from itertools import product
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix


In [ ]:
dataset = 'mnist'
n_samples = 2000
seed = 42
n_clusters = 10
latent_dim = 10
hidden_dims = [128, 64]
pretrain_epochs = 5
epochs = 20
batch_size = 256
lr = 1e-3
seed_everything(seed)
X, y = load_data(dataset, n_samples=n_samples, seed=seed)
dimensions = [X.shape[1], *hidden_dims, latent_dim]

run_loss_ablation = False
run_cluster_sweep = False
run_architecture_comparison = False
run_exclusive_ablation = False
run_binary_comparison = False
run_parameter_grid = False

result = train_vae_mf_model_optimized(
    X, n_clusters=n_clusters, dimensions=dimensions,
    pretrain_epochs=pretrain_epochs, epochs=epochs,
    batch_size=batch_size, lr=lr, seed=seed, y=y,
)
model, mf, history = result['model'], result['mf'], result['history']
pd.DataFrame([score_dict(y, result['pred'])])


## Curves

Loss plots include pretraining; metric histories start at the first joint epoch after pretraining. Smoothing is only for display; the original values remain visible.


In [ ]:
pretrain_history = result.get('pretrain_history', result.get('prehistory', {}))
full_history = {}
for key in ['loss', 'recon_loss', 'kl_loss', 'mf_loss']:
    before = pretrain_history.get(key, [0.] * len(pretrain_history.get('loss', [])))
    full_history[key] = before + history[key]
plot_history(full_history)
metric_names = ['ACC', 'NMI', 'ARI', 'F1', 'Purity']
frame = pd.DataFrame(history)
available = [name for name in metric_names if name in frame]
if available:
    ax = frame[available].plot(alpha=0.3, figsize=(8, 4), legend=False)
    frame[available].rolling(5, min_periods=1).mean().plot(ax=ax)
    ax.set(xlabel='Joint epoch', ylabel='Score', title='Metrics (5-epoch moving average)')
    plt.tight_layout()
    plt.show()

columns = [key for key in ['recon_loss', 'kl_loss', 'mf_loss'] if key in frame]
scaled = frame[columns].copy()
scaled = (scaled - scaled.min()) / (scaled.max() - scaled.min()).clip(lower=1e-12)
scaled.plot(figsize=(8, 4), title='Loss components, scaled separately for display')
plt.xlabel('Joint epoch')
plt.tight_layout()
plt.show()


## Latent space and confusion matrix


In [ ]:
def compute_tsne(Z, n_samples=1500, seed=42):
    indices = np.random.default_rng(seed).choice(len(Z), min(len(Z), n_samples), replace=False)
    selected = np.asarray(Z)[indices]
    if selected.shape[1] == 2:
        return selected, indices
    coords = TSNE(n_components=2, perplexity=min(30, max(2, (len(indices)-1)/3)),
                  init='pca', learning_rate='auto', random_state=seed).fit_transform(selected)
    return coords, indices


def plot_corrected_confusion_matrix(y, pred, normalize=False):
    matched = aligned_labels(y, pred)
    classes = np.unique(y)
    # Keep an unmatched column when k is larger than the number of true classes.
    other = [value for value in np.unique(matched) if value not in classes]
    labels = np.r_[classes, other]
    cm = confusion_matrix(y, matched, labels=labels)[:len(classes)]
    if normalize:
        cm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap='Blues')
    ax.set(xticks=range(len(labels)), yticks=range(len(classes)),
           xticklabels=[str(v) if v in classes else 'unmatched' for v in labels],
           yticklabels=classes, xlabel='Matched cluster', ylabel='True label')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    return fig

coords, indices = compute_tsne(result['Z'], seed=seed)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, labels, title in zip(axes, [y[indices], result['pred'][indices]], ['True labels', 'Cluster assignments']):
    ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap='tab10', s=5)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()
plot_corrected_confusion_matrix(y, result['pred'])
plt.show()


## Reconstructions and a 2D decoder grid

The grid trains a separate two-dimensional VAE. It does not pass two coordinates into the ten-dimensional model.


In [ ]:
def plot_latent_grid(model, side, grid_size=8, span=(-3, 3)):
    values = np.linspace(*span, grid_size)
    points = torch.tensor([[a, b] for b in values[::-1] for a in values],
                          dtype=torch.float32, device=device)
    model.eval()
    with torch.no_grad():
        images = model.decoder(points).cpu().reshape(grid_size, grid_size, side, side)
    canvas = images.permute(0, 2, 1, 3).reshape(grid_size*side, grid_size*side)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(canvas, cmap='gray', extent=[*span, *span])
    ax.set(xlabel='z1', ylabel='z2')
    return fig

side = int(np.sqrt(X.shape[1]))
if side * side == X.shape[1]:
    sample = X[:8].to(device)
    model.eval()
    with torch.no_grad():
        recon = model.decoder(model.encode(sample)).cpu()
    fig, axes = plt.subplots(2, 8, figsize=(10, 3))
    for i in range(len(sample)):
        axes[0, i].imshow(sample[i].cpu().reshape(side, side), cmap='gray')
        axes[1, i].imshow(recon[i].reshape(side, side), cmap='gray')
    for ax in axes.ravel():
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    seed_everything(seed)
    vae2 = VariationalAutoEncoder([X.shape[1], *hidden_dims, 2]).to(device)
    pre_train(X, vae2, epochs=pretrain_epochs, batch_size=batch_size, lr=lr, seed=seed)
    plot_latent_grid(vae2, side)
    plt.show()


## Four loss settings (Section 6.3)

Reconstruction only, KL only, reconstruction with KL warm-up, and the full joint model. The KL-only run is a diagnostic of representation collapse. It deliberately has no reconstruction term. All four models use the same initial weights and total number of epochs. The joint model makes two network updates per batch; the other settings make one. Each t-SNE is fitted separately, so positions cannot be compared across panels.


In [ ]:
def train_loss_ablation(X, initial_model, setting, epochs=20, batch_size=256, lr=1e-3, seed=42):
    seed_everything(seed)
    model = deepcopy(initial_model).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=True)
    for epoch in range(epochs):
        model.train()
        beta = min(1.0, 0.1 + 0.9 * epoch / max(1, epochs-1))
        for (batch,) in loader:
            batch = batch.to(device)
            recon, mu, log_var = model(batch)
            recon_loss, kl_loss = Loss_Function(recon, batch, mu, log_var)
            if setting == 'reconstruction':
                loss = recon_loss
            elif setting == 'kl_only':
                loss = kl_loss
            elif setting == 'kl_warmup':
                loss = recon_loss + beta * kl_loss
            else:
                raise ValueError(setting)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return vae_encode(X, model, batch_size)

if run_loss_ablation:
    seed_everything(seed)
    initial_model = VariationalAutoEncoder(dimensions).to(device)
    embeddings = {}
    for setting in ['reconstruction', 'kl_only', 'kl_warmup']:
        embeddings[setting] = train_loss_ablation(X, initial_model, setting,
            epochs=pretrain_epochs+epochs, batch_size=batch_size, lr=lr, seed=seed)
    joint = train_vae_mf_model_optimized(X, n_clusters=n_clusters,
        model=deepcopy(initial_model), pretrain_epochs=pretrain_epochs, epochs=epochs,
        batch_size=batch_size, lr=lr, seed=seed)
    embeddings['joint'] = joint['Z']
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for ax, (name, Z) in zip(axes.ravel(), embeddings.items()):
        coords, indices = compute_tsne(Z, seed=seed)
        ax.scatter(coords[:, 0], coords[:, 1], c=y[indices], s=4, cmap='tab10')
        ax.set_title(name)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()


## Number of clusters and architecture

The cluster sweep covers $k=2,\ldots,18$. ACC uses one-to-one Hungarian matching, so surplus clusters remain unmatched. Purity has a different interpretation and can increase with k. The convolutional comparison is for 28×28 images; the thesis did not preserve a Reuters reshape protocol.


In [ ]:
if run_cluster_sweep:
    rows = []
    for k in range(2, 19):
        current = train_vae_mf_model_optimized(X, n_clusters=k, dimensions=dimensions,
            pretrain_epochs=pretrain_epochs, epochs=epochs, batch_size=batch_size,
            lr=lr, seed=seed, verbose=False)
        rows.append({'k': k, **score_dict(y, current['pred'])})
    k_scores = pd.DataFrame(rows)
    display(k_scores)
    k_scores.plot(x='k', y=metric_names, marker='o', figsize=(8, 4))
    plt.ylabel('Score')
    plt.tight_layout()
    plt.show()

if run_architecture_comparison:
    rows = []
    for architecture in ['fc', 'conv']:
        seed_everything(seed)
        current_model = (VariationalAutoEncoder(dimensions) if architecture == 'fc'
                         else ConvolutionalVAE(input_dim=X.shape[1], latent_dim=latent_dim))
        current = train_vae_mf_model_optimized(X, n_clusters=n_clusters,
            model=current_model.to(device), pretrain_epochs=pretrain_epochs,
            epochs=epochs, batch_size=batch_size, lr=lr, seed=seed, verbose=False)
        rows.append({'architecture': architecture, **score_dict(y, current['pred'])})
    display(pd.DataFrame(rows))


## Exclusive Lasso (Section 6.7)

Freeze the VAE representation and disable binary updates to isolate the exclusive penalty. Each condition starts from the same Y and C. GDR is the mean row maximum divided by its L1 sum; an all-zero row contributes zero.


In [ ]:
if run_exclusive_ablation:
    Z = result['Z']
    generator = np.random.default_rng(seed)
    Y0 = generator.uniform(size=(len(X), n_clusters)).astype('float32')
    C0 = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit(Z).cluster_centers_
    lasso_histories = {}
    for mode in ['none', 'thesis', 'exact']:
        current = train_mf_binary_model(Z, n_clusters=n_clusters, epochs=epochs,
            gamma=1e-3, binary_mode='none', exclusive_mode=mode, seed=seed,
            Y_init=Y0, C_init=C0)
        lasso_histories[mode] = pd.DataFrame(current['history'])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for mode, frame in lasso_histories.items():
        axes[0].plot(frame['GDR'], label=mode)
        axes[1].plot(frame['zero_count'], label=mode)
    for ax, ylabel in zip(axes, ['Mean group dominance ratio', 'Near-zero elements in Y']):
        ax.set(xlabel='Epoch', ylabel=ylabel)
        ax.legend()
    plt.tight_layout()
    plt.show()


## Binary updates and parameter search

The earlier notebooks tried several binary mappings and parameter grids. These comparisons retain all scores. Labels are used for reporting only; there is no ACC-based stopping or weight adjustment. Comparing losses from different regularization weights is not a valid way to rank clustering quality.


In [ ]:
if run_binary_comparison:
    rows = []
    for mode in ['none', 'primp', 'improved', 'legacy', 'legacy_alt']:
        current = train_mf_binary_model(result['Z'], n_clusters=n_clusters, epochs=epochs,
            kappa=1e-3, gamma=0, binary_mode=mode, exclusive_mode='none', seed=seed)
        rows.append({'binary_mode': mode, **score_dict(y, current['pred']),
                     **assignment_diagnostics(current['mf'].Y.detach())})
    display(pd.DataFrame(rows))


def grid_search_train_vae_mf_model(X, y, dimensions, param_grid, seed=42):
    rows = []
    for values in product(*param_grid.values()):
        parameters = dict(zip(param_grid, values))
        current = train_vae_mf_model_optimized(X, n_clusters=n_clusters, dimensions=dimensions,
            pretrain_epochs=pretrain_epochs, epochs=epochs, batch_size=batch_size,
            seed=seed, verbose=False, **parameters)
        rows.append({**parameters, 'final_loss': current['history']['loss'][-1],
                     **score_dict(y, current['pred'])})
    return pd.DataFrame(rows)

if run_parameter_grid:
    # Edit this ordinary dictionary to try other parameters.
    param_grid = {'lr': [1e-4, 1e-3], 'kappa': [1e-4, 1e-3], 'gamma': [0, 1e-3]}
    grid_scores = grid_search_train_vae_mf_model(X, y, dimensions, param_grid, seed)
    display(grid_scores)
